# CS 195: Natural Language Processing
## Review of Logistic Regression and Optimization, Introduction to PyTorch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmanley/s26-CS195NLP/blob/main/F3_2_LogisticRegressionPyTorch.ipynb)


## References

[SLP: Logistic Regression and Text Classification, Chapter 4 of Speech and Language Processing by Daniel Jurafsky & James H. Martin](https://web.stanford.edu/~jurafsky/slp3/4.pdf)


<img src="https://github.com/ericmanley/s26-CS195NLP/blob/main/images/key_disable.png?raw=1" />

In [ ]:
#import sys
#!{sys.executable} -m pip install datasets transformers torch

## Review: Integer Encoding

We tried machine learning with text where each word was assigned a number - integer encoding

We have to make sure each input has the same size. Since text inputs are different sizes
* pad small ones with zeros
* truncate long ones

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.metrics import accuracy_score
from sklearn import tree

dt = tree.DecisionTreeClassifier(random_state=41)

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM3-3B")

data = load_dataset("Deysi/spam-detection-dataset")


train_encoding = tokenizer(list(data["train"]["text"]),truncation=True,padding="max_length",max_length=512)
train_labels = data["train"]["label"]
test_encoding = tokenizer(list(data["test"]["text"]),truncation=True,padding="max_length",max_length=512)
test_labels = data["test"]["label"]


dt.fit(train_encoding["input_ids"],train_labels)

predictions = dt.predict(test_encoding["input_ids"])

print( accuracy_score(test_labels,predictions) )

0.8928440366972477


## Review: Bag-of-Words Encoding

Choose vocabulary (say 5000 most common words) one column for each word

row contains counts for each word

**Example**

*Sentence 1:* "The cat sat on the hat"

*Sentence 2:* "The dog ate the cat and the hat"

*Vocabulary:* { the, cat, sat, on, hat, dog, ate, and }


|            | the | cat | sat | on | hat | dog | ate | and |
|------------|-----|-----|-----|----|-----|-----|-----|-----|
| Sentence 1 | 2   | 1   | 1   | 1  | 1   | 0   | 0   | 0   |
| Sentence 2 | 3   | 1   | 0   | 0  | 1   | 1   | 1   | 1   |


**The downside:** this doesn't maintain any information about word order - thus the "bag" of words

`scikit-learn` provides a Bag-of-Words encoder called `CountVectorizer`


In [ ]:
from sklearn import tree
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import CountVectorizer

data = load_dataset("Deysi/spam-detection-dataset")

train_texts = data["train"]["text"]
train_labels = data["train"]["label"]
test_texts = data["test"]["text"]
test_labels = data["test"]["label"]

# Consider top 5000 frequent words
# remove stop words
vectorizer = CountVectorizer(max_features=5000,stop_words="english")
vectorizer.fit(train_texts)

train_vectors = vectorizer.transform(train_texts)
test_vectors = vectorizer.transform(test_texts)

dt = tree.DecisionTreeClassifier(random_state=41)
dt.fit(train_vectors,train_labels)

predictions = dt.predict(test_vectors)

print( accuracy_score(test_labels,predictions) )

0.9746788990825688


## TD-IDF Encoding

**TF-IDF:** Term Frequency - Inverse Document Frequency

**Term Frequency:** How often does the word appear in the example, like CountVectorizer
* actually take the $\log$ of it

**Document Frequency:** What fraction of the *documents* (or *training-examples*) does the word appear in?

**Inverse Document Frequency:** (number of documents) / (number of documents containing the word)
* *I forgot to mention this last time: we take the log of this too* - measuring in orders of magnitude helps keep this from dominating the product
* if a word is in only a few documents, you get a big number
* if a word appears in lots of documents, you get a small number

When encoding a new example, multiply the Term Frequency of the word in this example by the Inverse Document Frequency of the training set
* gives higher weight to words that are differentiators
* stop words should automatically be de-emphasized

**Example:**
Document collection: all of Shakespeare's plays

The word `Romeo` appears 113 times but only in 1 document

The word `action` appears 113 time but in 31 documents

so Romeo will get a much higher weight


In [ ]:
from sklearn import tree
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer

data = load_dataset("Deysi/spam-detection-dataset")

train_texts = data["train"]["text"]
train_labels = data["train"]["label"]
test_texts = data["test"]["text"]
test_labels = data["test"]["label"]

# Consider top 5000 frequent words
vectorizer = TfidfVectorizer(max_features=5000)
vectorizer.fit(train_texts)

train_vectors = vectorizer.transform(train_texts)
test_vectors = vectorizer.transform(test_texts)

dt = tree.DecisionTreeClassifier(random_state=41)
dt.fit(train_vectors,train_labels)

predictions = dt.predict(test_vectors)

print( accuracy_score(test_labels,predictions) )

0.9842201834862385


In [ ]:
type(train_vectors)

scipy.sparse._csr.csr_matrix

In [ ]:
type(test_labels)

datasets.arrow_dataset.Column

## PyTorch introduction

PyTorch is the go-to library for neural network models in Python

We're going to start with a simple model - we'll review Logistic Regression and see how it works in PyTorch

Let's start with the setup. First the libraries we need and loading the dataset as before

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.feature_extraction.text import TfidfVectorizer
from datasets import load_dataset


data = load_dataset("Deysi/spam-detection-dataset")

train_texts = data["train"]["text"]
test_texts  = data["test"]["text"]

vectorizer = TfidfVectorizer(max_features=5000)
train_vectors = vectorizer.fit_transform(train_texts)
test_vectors  = vectorizer.transform(test_texts)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/581 [00:00<?, ?B/s]

data/train-00000-of-00001-daf190ce720b3d(…):   0%|          | 0.00/1.92M [00:00<?, ?B/s]

data/test-00000-of-00001-fa9b3e8ade89a33(…):   0%|          | 0.00/663k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8175 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2725 [00:00<?, ? examples/s]

but we need numerical values for the labels rather than strings like sklearn is ok with

In [ ]:
train_labels = [1 if y == "spam" else 0 for y in data["train"]["label"]]
test_labels  = [1 if y == "spam" else 0 for y in data["test"]["label"]]

### Tensors

PyTorch has its own data structure, `tensor`, for storing arrays/vectors.

The sklearn vectorizer returns a `scipy` based matrix that has to be converted into a `numpy` array before it can be converted into a `tensor`

The label lists we made can be converted directly, but we need to `unsqueeze` it by one dimension, meaning turn a 1 dimensional tensor into a 2-dimensional one - it's like putting a list into another list where it is the only element

In [ ]:
type(train_vectors)

scipy.sparse._csr.csr_matrix

In [ ]:
X_train = torch.tensor(train_vectors.toarray(), dtype=torch.float32)
X_test = torch.tensor(test_vectors.toarray(), dtype=torch.float32)
X_train

tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.1507, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]])

In [ ]:
y_train = torch.tensor(train_labels, dtype=torch.float32).unsqueeze(1)
y_test = torch.tensor(test_labels, dtype=torch.float32).unsqueeze(1)
y_train

tensor([[0.],
        [1.],
        [1.],
        ...,
        [0.],
        [1.],
        [1.]])

## Logistic Regression

Critical assumption with logistic regression: the real phenomenon can be modeled by this linear function

$$y = w_nx_n+w_{n-1}x_{n-1}+\cdots+w_1x_1+b$$

The $x$s: numeric inputs from Bag of Words or whatever encoding you're using

The $w$s and $b$: some weights/parameters that we learn

Positive $y$: predict True

Negative $y$: predict False

### Vector representation

This equation is often described as vectors, with the parameter vector $\boldsymbol{w} = [w_1, w_2, \ldots, w_n]$ and input vector $\boldsymbol{x} = [x_1, x_2, \ldots, x_n]$ and we use the dot product

$$y = \boldsymbol{w} \cdot \boldsymbol{x} + \boldsymbol{b}$$

### Squashing Function

We usually take the output and put it through a *squashing*/*activation* functino like the **sigmoid** function $\sigma(y) = 1/(1+e^{-y})$

<center><img src = "images/sigmoid.png" /></center>

image credit: [SLP Fig 4.1](https://web.stanford.edu/~jurafsky/slp3/4.pdf)

**Why?**
* takes big numbers and forces them to be between 0 and 1
* very close to 0 and 1 most of the time, but it looks linear near the center when we're uncertain
* differentiable, so it works with all the calculus


## Linear Model representation in PyTorch

What does this model looks like in PyTorch?
* Remember our TF-IDF vectors were set to use 5000 tokens
* We have only a single linear node

In [ ]:
model = torch.nn.Linear(5000, 1)

### Discussion Question

This works because we only have a single binary answer

How does a linear separator work if you have 3 or more classes to separate?

What do you think you'd do differently in the PyTorch code?

## Learning with Logistic Regression

To learn the right $y = \boldsymbol{w} \cdot \boldsymbol{x} + \boldsymbol{b}$, we use some kind of optimization algorithm

Example: Gradient Descent
* Quick explanation and visualization: https://www.youtube.com/watch?v=qg4PchTECck
* More visualization: https://www.youtube.com/embed/GkB4vW16QHI?si=7BpkMgIqIaLXM89-&amp;start=126


### Mental hang-up

Gradient Descent is often shown as trying to find the point on a surface that is smallest, but we're **not** trying to find the minimum $\boldsymbol{x}$ that minimizes $y = \boldsymbol{w} \cdot \boldsymbol{x} + \boldsymbol{b}$.

We're trying to find a $\boldsymbol{w}$ and $\boldsymbol{b}$ that minimizes how different our guess/model $$\hat{y} = \boldsymbol{w} \cdot \boldsymbol{x} + \boldsymbol{b}$$ is from the real $y$ given all the training examples $\boldsymbol{x}$ that we have.

But the same algorithm still works!

### Loss functions

The way you measure how different your model $\hat{y}$ is from reality $y$ is called the **loss function**
* there are many you could choose

For categorical data, we usually use the **cross-entropy loss** (also called **negative log likelihood loss**) which looks like this

$$L_{CE} = -[y\log(\hat{y})+(1-y)\log(1-\hat{y})]$$

**Where does this come from?**

Let's say we see a given input $x$ - remember this is like the bag-of-words representation (or some other encoding) of some text

and we're trying to find the weights that give us the correct label $y$ (i.e., spam or not-spam)

The *probability* that $y$ is the right label given that we've observed input $x$ is

$$p(y|x) = \hat{y}^y{(1-\hat{y})}^{(1-y)}$$

Why?

* if $y=1$, then $\hat{y}^y{(1-\hat{y})}^{(1-y)}$ simplifies to $\hat{y}$
  * $p(1|x) = \hat{y}$ if the model guessed $\hat{y}=1$, then we got it right - a high probability
  * $p(1|x) = \hat{y}$ if the model guessed $\hat{y}=0$, then we got it wrong - a low probability

    
* if $y=0$, then $\hat{y}^y{(1-\hat{y})}^{(1-y)}$ simplifies to $1-\hat{y}$
  * $p(0|x) = 1-\hat{y}$ if the model guessed $\hat{y}=0$, then we got it right - a high probability
  * $p(0|x) = 1-\hat{y}$ if the model guessed $\hat{y}=1$, then we got it wrong - a low probability

Takeaway:
* when the model assigns a guess with high confidence ($\hat{y}$ close to 0 or 1) and is correct, the $p(y|x)$ expression is high
* when the model assigns a guess with high confidence ($\hat{y}$ close to 0 or 1) and is wrong, the $p(y|x)$ expression is low

#### However

the cross-entropy loss function actually uses the negative log of this $$-\log(p(y|x)) = -[y\log(\hat{y})+(1-y)\log(1-\hat{y})]$$

Why?
* turns products into sums - repeated products less than 1 get small quickly and are hard to work with
* minimize instead of maximize - smaller loss is easier to think of as being more correct, just like with mean-squared-error, you want to make it small
* when used to make changes to the weights (the *gradients*), big errors (confident and wrong) cause significantly bigger changes than small errors




### So how much do we change the weight based on this loss function?

Think of moving along each dimension independently, and we get the direction and magnitude that each weight needs to move by taking the partial derivative with respect to that dimension:

$$\frac{\partial L_{CE}(\hat{y},{y})}{\partial w_j} = (\text{do some calculus}) = (\hat{y}-y)x_j$$

The partial derivative calculates the slope along that dimension as we move towards the minimum loss - if the slope is big then we want to nudge the weight a lot in that direction, if it is small, we only want to nudge a little

**do some calculus** means plug in $y$ (right answer from training set) and $\hat{y}$ (what the model guessed) to the loss function, take the derivative with respect to this weight, and after applying the chain rule, you get $(\hat{y}-y)x_j$

This value, $(\hat{y}-y)x_j$ is called the **gradient**

except, we can also control how quickly allow the weights to change by multiplying by a small *learning rate* $\eta$

so

the new weight is $$w_j \leftarrow w_j - \eta(\hat{y}-y)x_j$$

## Loss Function and Optimizer Algorithm in PyTorch

`BCEWithLogitsLoss` is PyTorch's implementation of cross-entropy loss *when you have only two possible classes*, i.e., Binary Cross Entropy
* *logits* means the score of $y = \boldsymbol{w} \cdot \boldsymbol{x} + \boldsymbol{b}$ before squashing with the sigmoid function. This loss function will apply the sigmoid function itself, so it wants you to *input* the logits rather than the already-squashed version

`SGD` is PyTorch's implementation of gradient descent that uses this update rule
* `lr` is the learning rate, $\eta$ that we set

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

### Discussion Question

What if you have more than two categories (think an emotions dataset rather than spam/not-spam

Do some searching and see if you can find out what you're supposed to use in this case instead

## PyTorch Training Loop

In [ ]:
for epoch in range(20):
    # clear stored gradients
    optimizer.zero_grad()

    # get predictions on the training examples
    logits = model(X_train)

    # calculate the loss - how wrong are the predictions?
    loss = loss_fn(logits, y_train)

    # calculate all the gradients for each parameter
    loss.backward()

    # update weights using those gradients
    optimizer.step()

    print(f"Epoch {epoch+1}, loss = {loss.item():.4f}")

Epoch 1, loss = 0.5849
Epoch 2, loss = 0.5845
Epoch 3, loss = 0.5840
Epoch 4, loss = 0.5836
Epoch 5, loss = 0.5832
Epoch 6, loss = 0.5828
Epoch 7, loss = 0.5823
Epoch 8, loss = 0.5819
Epoch 9, loss = 0.5815
Epoch 10, loss = 0.5811
Epoch 11, loss = 0.5807
Epoch 12, loss = 0.5802
Epoch 13, loss = 0.5798
Epoch 14, loss = 0.5794
Epoch 15, loss = 0.5790
Epoch 16, loss = 0.5786
Epoch 17, loss = 0.5781
Epoch 18, loss = 0.5777
Epoch 19, loss = 0.5773
Epoch 20, loss = 0.5769


Let's break this down

each parameter `param` in the model has
* `param.data` the current weight value
* `param.grad` the accumulated gradient

`optimizer.zero_grad()` zeros `param.grad` for all parameters

`loss.backward()` accumulates new values for each `param.grad` based on the loss just calculated

`optimizer.step()` updates `param.data` using `param.grad`

We could wait to do `optimizer.step()` until after doing more accumulations - this is the difference between different variations of gradient descent, like stochastic gradient descent and batch gradient descent


## Evaluation Loop

When doing evaluations, you can run `torch.no_grad()` to tell PyTorch not to track any gradients inside this block of code


In [ ]:
with torch.no_grad():
    predictions = torch.sigmoid(model(X_test))
    predicted_labels = (predictions > 0.5).int()
    accuracy = (predicted_labels == y_test.int()).float().mean()
    print(predictions)
    print(predicted_labels)
    print(y_test)
    print(accuracy)

print("Accuracy:", accuracy.item())

tensor([[0.4547],
        [0.6201],
        [0.4448],
        ...,
        [0.4276],
        [0.4770],
        [0.5965]])
tensor([[0],
        [1],
        [0],
        ...,
        [0],
        [0],
        [1]], dtype=torch.int32)
tensor([[0.],
        [1.],
        [0.],
        ...,
        [0.],
        [0.],
        [1.]])
tensor(0.9886)
Accuracy: 0.988623857498169


### Group Investigation

Does the accuracy improve with additional runs of the training loop?

print out `predictions` and `predictions>0.5` and `(predictions > 0.5).int()` what's going on here?

print out `y_test` and `predicted_labels == y_test.int()` what's going on here?

What happens if I print `accuracy` instead of `accuracy.item()`?

## Putting it all together

Here's the full PyTorch workflow for this problem

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.feature_extraction.text import TfidfVectorizer
from datasets import load_dataset


data = load_dataset("Deysi/spam-detection-dataset")

train_texts = data["train"]["text"]
test_texts  = data["test"]["text"]

vectorizer = TfidfVectorizer(max_features=5000)
train_vectors = vectorizer.fit_transform(train_texts)
test_vectors  = vectorizer.transform(test_texts)

train_labels = [1 if y == "spam" else 0 for y in data["train"]["label"]]
test_labels  = [1 if y == "spam" else 0 for y in data["test"]["label"]]

X_train = torch.tensor(train_vectors.toarray(), dtype=torch.float32)
X_test = torch.tensor(test_vectors.toarray(), dtype=torch.float32)

y_train = torch.tensor(train_labels, dtype=torch.float32).unsqueeze(1)
y_test = torch.tensor(test_labels, dtype=torch.float32).unsqueeze(1)

model = torch.nn.Linear(5000, 1)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(100):
    optimizer.zero_grad()

    logits = model(X_train)
    loss = loss_fn(logits, y_train)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, loss = {loss.item():.4f}")


with torch.no_grad():
    predictions = torch.sigmoid(model(X_test))
    predicted_labels = (predictions > 0.5).int()
    accuracy = (predicted_labels == y_test.int()).float().mean()
    print(accuracy)

print("Accuracy:", accuracy.item())

Epoch 1, loss = 0.6940
Epoch 2, loss = 0.6934
Epoch 3, loss = 0.6928
Epoch 4, loss = 0.6923
Epoch 5, loss = 0.6917
Epoch 6, loss = 0.6911
Epoch 7, loss = 0.6905
Epoch 8, loss = 0.6899
Epoch 9, loss = 0.6894
Epoch 10, loss = 0.6888
Epoch 11, loss = 0.6882
Epoch 12, loss = 0.6877
Epoch 13, loss = 0.6871
Epoch 14, loss = 0.6865
Epoch 15, loss = 0.6860
Epoch 16, loss = 0.6854
Epoch 17, loss = 0.6848
Epoch 18, loss = 0.6843
Epoch 19, loss = 0.6837
Epoch 20, loss = 0.6832
Epoch 21, loss = 0.6826
Epoch 22, loss = 0.6820
Epoch 23, loss = 0.6815
Epoch 24, loss = 0.6809
Epoch 25, loss = 0.6804
Epoch 26, loss = 0.6798
Epoch 27, loss = 0.6793
Epoch 28, loss = 0.6787
Epoch 29, loss = 0.6782
Epoch 30, loss = 0.6776
Epoch 31, loss = 0.6771
Epoch 32, loss = 0.6765
Epoch 33, loss = 0.6760
Epoch 34, loss = 0.6754
Epoch 35, loss = 0.6749
Epoch 36, loss = 0.6743
Epoch 37, loss = 0.6738
Epoch 38, loss = 0.6733
Epoch 39, loss = 0.6727
Epoch 40, loss = 0.6722
Epoch 41, loss = 0.6716
Epoch 42, loss = 0.6711
E

## Applied Exploration

Select another Hugging Face dataset for text classification and get it working with this code.

Note, that if you use a dataset with more than 2 classes (a laudable goal to get working)
* You will need to change the `model` to match
* You need to find the non-binary version of cross-entropy for the loss function
* You may need format for the labels
* When evaluating, you will need to look at what the predictions look like and compute accordingly

Give a short write-up on the following
* Describe your dataset, including the distribution of the target variable
* Interpret the results - How did this dataset compare with the spam dataset? Why do you think you got the results that you did?

Language Identification
[papluca/language-identification](https://huggingface.co/datasets/papluca/language-identification)

The Language Identification dataset is a collection of 90k samples consisting of text passages and corresponding language label. This dataset was created by collecting data from 3 sources: Multilingual Amazon Reviews Corpus, XNLI, and STSb Multi MT.

The Language Identification dataset contains text in 20 languages, which are:

arabic (ar), bulgarian (bg), german (de), modern greek (el), english (en), spanish (es), french (fr), hindi (hi), italian (it), japanese (ja), dutch (nl), polish (pl), portuguese (pt), russian (ru), swahili (sw), thai (th), turkish (tr), urdu (ur), vietnamese (vi), and chinese (zh)

Dataset Structure
Data Instances For each instance, there is a string for the text and a string for the label (the language tag). Here is an example:

{'labels': 'fr', 'text': 'Conforme à la description, produit pratique.'}

Data Fields labels: a string indicating the language label. text: a string consisting of one or more sentences in one of the 20 languages listed above. Data Splits The Language Identification dataset has 3 splits: train, valid, and test. The train set contains 70k samples, while the validation and test sets 10k each. All splits are perfectly balanced: the train set contains 3500 samples per language, while the validation and test sets 500.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.feature_extraction.text import TfidfVectorizer
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder

# Load dataset
data = load_dataset("papluca/language-identification")

train_texts = data["train"]["text"]
test_texts  = data["test"]["text"]

# TF-IDF features
vectorizer = TfidfVectorizer(max_features=5000)
train_vectors = vectorizer.fit_transform(train_texts)
test_vectors  = vectorizer.transform(test_texts)

# Convert labels to integers (0-19)
encoder = LabelEncoder()
train_labels = encoder.fit_transform(data["train"]["labels"])
test_labels  = encoder.transform(data["test"]["labels"])

# Convert to torch tensors
X_train = torch.tensor(train_vectors.toarray(), dtype=torch.float32)
X_test = torch.tensor(test_vectors.toarray(), dtype=torch.float32)

y_train = torch.tensor(train_labels, dtype=torch.long)
y_test = torch.tensor(test_labels, dtype=torch.long)

num_classes = len(encoder.classes_)  # should be 20
print(num_classes)

# Model: 5000 input features → 20 outputs
model = nn.Linear(5000, num_classes)

# Multiclass loss
loss_fn = nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(), lr=0.1)

# Training loop
for epoch in range(50):
    optimizer.zero_grad()

    logits = model(X_train)
    loss = loss_fn(logits, y_train)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, loss = {loss.item():.4f}")

# Evaluation
with torch.no_grad():
    logits = model(X_test)
    predicted_labels = torch.argmax(logits, dim=1)
    accuracy = (predicted_labels == y_test).float().mean()

print("Accuracy:", accuracy.item())

20
Epoch 1, loss = 2.9962
Epoch 2, loss = 2.9958
Epoch 3, loss = 2.9954
Epoch 4, loss = 2.9951
Epoch 5, loss = 2.9947
Epoch 6, loss = 2.9943
Epoch 7, loss = 2.9939
Epoch 8, loss = 2.9935
Epoch 9, loss = 2.9931
Epoch 10, loss = 2.9927
Epoch 11, loss = 2.9923
Epoch 12, loss = 2.9920
Epoch 13, loss = 2.9916
Epoch 14, loss = 2.9912
Epoch 15, loss = 2.9908
Epoch 16, loss = 2.9904
Epoch 17, loss = 2.9900
Epoch 18, loss = 2.9896
Epoch 19, loss = 2.9892
Epoch 20, loss = 2.9889
Epoch 21, loss = 2.9885
Epoch 22, loss = 2.9881
Epoch 23, loss = 2.9877
Epoch 24, loss = 2.9873
Epoch 25, loss = 2.9869
Epoch 26, loss = 2.9865
Epoch 27, loss = 2.9861
Epoch 28, loss = 2.9858
Epoch 29, loss = 2.9854
Epoch 30, loss = 2.9850
Epoch 31, loss = 2.9846
Epoch 32, loss = 2.9842
Epoch 33, loss = 2.9838
Epoch 34, loss = 2.9834
Epoch 35, loss = 2.9830
Epoch 36, loss = 2.9827
Epoch 37, loss = 2.9823
Epoch 38, loss = 2.9819
Epoch 39, loss = 2.9815
Epoch 40, loss = 2.9811
Epoch 41, loss = 2.9807
Epoch 42, loss = 2.980

## Model Setup

Input features: TF-IDF vectors, 5000 features

Model: Linear layer 5000 → 20 (one output per language)

Loss: CrossEntropyLoss (non-binary)

Optimizer: SGD, lr=0.1

Epochs: 50

## first Results

Training loss: slowly decreased (from 2.996 → 2.977)

Test accuracy: 50.0%


# Interpretation

Compared to the spam dataset:

Spam dataset was binary, easy to separate → accuracy >+80% quickly.

Language identification is 20-class, much harder.

TF-IDF with only 5000 word features loses a lot of character-level information, which is key for distinguishing similar languages (e.g., Spanish vs Portuguese).

Why results are lower than expected:

TF-IDF uses word-level features — for short text, character n-grams are more informative.

Simple linear model is underpowered for 20-class classification.

SGD with fixed learning rate may converge slowly for this dataset.

## Reasons on why the lower accuracy

Character set and tokenization differences

Languages like Chinese, Arabic, Hindi, Japanese use non-Latin scripts.

TF-IDF vectorizer with default settings assumes Latin word boundaries, so it may split these languages poorly or ignore important characters.

Example: "مرحبا" (Arabic) may become a single token

## Printing out missed

In [ ]:

for i, label in enumerate(encoder.classes_[:20]):
    print(f"Index: {i}, Language: {label}")

Index: 0, Language: ar
Index: 1, Language: bg
Index: 2, Language: de
Index: 3, Language: el
Index: 4, Language: en
Index: 5, Language: es
Index: 6, Language: fr
Index: 7, Language: hi
Index: 8, Language: it
Index: 9, Language: ja
Index: 10, Language: nl
Index: 11, Language: pl
Index: 12, Language: pt
Index: 13, Language: ru
Index: 14, Language: sw
Index: 15, Language: th
Index: 16, Language: tr
Index: 17, Language: ur
Index: 18, Language: vi
Index: 19, Language: zh


In [ ]:
predicted_labels_str = encoder.inverse_transform(predicted_labels.numpy())

# Print top 10 misclassified examples
count = 0
max_print = 10

for i in range(len(y_test)):
    true_label = data["test"]["labels"][i]          # actual language string
    pred_label = predicted_labels_str[i]            # predicted language string

    if pred_label != true_label:
        print(f"Text: {test_texts[i]}")
        print(f"True: {true_label}, Predicted: {pred_label}")
        print("---")

        count += 1
        if count >= max_print:
            break

Text: De technologisch geplaatste Nasdaq Composite Index .IXIC daalde met 25,36 punten, of 1,53 procent, tot 1.628,26.
True: nl, Predicted: pl
---
Text: Через каждые сто градусов пятна краски меняют свой цвет, она может быть красной и изменить цвет на синий.
True: ru, Predicted: de
---
Text: Verschillende mensen op motorfietsen op een marktplein.
True: nl, Predicted: ar
---
Text: No funciona lo he devuelto, no hace nada
True: es, Predicted: ar
---
Text: Debutta Apple iPad mini
True: it, Predicted: ur
---
Text: Szympans kopie cel.
True: pl, Predicted: de
---
Text: إذا يبين اللون الفاتح   ميزة للبيرا ( مثلا.
True: ar, Predicted: de
---
Text: 子育て中で抱っこをしていたら、毎日腰が痛すぎて買いました。 骨盤も締めれるので重宝してますが、マジックテープを剥がす音がうるさいので、子供が寝てる時は別室で剥がします。 サポート感はあるので、ずいぶん腰は楽になりました。 音だけ、改善してほしいです。
True: ja, Predicted: de
---
Text: 本体にぴったりサイズで、作りもしっかりしているようで安心感があります。
True: ja, Predicted: de
---
Text: Come si fa?
True: it, Predicted: ja
---


## Trying ADAM

In [ ]:
model = torch.nn.Linear(5000, 20)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(200):
    optimizer.zero_grad()
    logits = model(X_train)
    loss = loss_fn(logits, y_train)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch+1}, loss = {loss.item():.4f}")

with torch.no_grad():
    logits = model(X_test)
    predicted_labels = logits.argmax(dim=1)
    accuracy = (predicted_labels == y_test).float().mean()

print("Accuracy:", accuracy.item())

Epoch 1, loss = 2.9950
Epoch 21, loss = 2.9061
Epoch 41, loss = 2.8180
Epoch 61, loss = 2.7311
Epoch 81, loss = 2.6454
Epoch 101, loss = 2.5611
Epoch 121, loss = 2.4784
Epoch 141, loss = 2.3974
Epoch 161, loss = 2.3182
Epoch 181, loss = 2.2410
Accuracy: 0.92330002784729


In [ ]:
predicted_labels_str = encoder.inverse_transform(predicted_labels.numpy())

# Print top 10 misclassified examples
count = 0
max_print = 10

for i in range(len(y_test)):
    true_label = data["test"]["labels"][i]          # actual language string
    pred_label = predicted_labels_str[i]            # predicted language string

    if pred_label != true_label:
        print(f"Text: {test_texts[i]}")
        print(f"True: {true_label}, Predicted: {pred_label}")
        print("---")

        count += 1
        if count >= max_print:
            break

Text: Debutta Apple iPad mini
True: it, Predicted: de
---
Text: Szympans kopie cel.
True: pl, Predicted: zh
---
Text: إذا يبين اللون الفاتح   ميزة للبيرا ( مثلا.
True: ar, Predicted: zh
---
Text: 子育て中で抱っこをしていたら、毎日腰が痛すぎて買いました。 骨盤も締めれるので重宝してますが、マジックテープを剥がす音がうるさいので、子供が寝てる時は別室で剥がします。 サポート感はあるので、ずいぶん腰は楽になりました。 音だけ、改善してほしいです。
True: ja, Predicted: zh
---
Text: 本体にぴったりサイズで、作りもしっかりしているようで安心感があります。
True: ja, Predicted: zh
---
Text: これからの季節のデスクワーク用に購入しました。 箱が安っぽくて最初心配しましたが、中身はしっかりしてました（笑） スタンドが小さく感じたのですが、本体が軽いので置く場所が平らであれば充電ケーブルを接続してもしっかり直立してくれます。 風も3段階調整でき、強にするとかなり涼しいです。
True: ja, Predicted: zh
---
Text: 家族が使用していますが、軽くて使いやすいと好評です。 頭皮に当たる部分もソフトな触り心地で嫌な感じは全くないと言っています。 大きさも適当なサイズかと思います。
True: ja, Predicted: zh
---
Text: Este é um pedido muito invulgar.
True: pt, Predicted: es
---
Text: 気になっていたこの枕を購入。 イメージ通りの寝心地の良さです。 ふわふわしてとても良いです。枕 を変えた当日は違和感が有り眠れなかったですが、 ぐっすり眠れるようになりました。
True: ja, Predicted: zh
---
Text: Здание суда - не единственный политический цирк сегодня утром в Вашингтоне.
True: ru, Predict

The biggest difference im noticing is that without adam its guesses tend to be all over the place
with adam it struggling on languages with very similar dialect.
like

spanish and portugeuese

russian and bulgarian

chinese and japanese